<a href="https://colab.research.google.com/github/vlgul/series-de-tiempo/blob/main/Capitulo_8_Tema_8_1_Suavizacion_exponencial_simple.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Capitulo 8:  Suavización exponencial** ###

### **8.1. Suavización exponencial simple** ###

Considere los datos de la pesca de bacalao del Ejemplo 6.1., los cuales se proporcionan en la tabla 6.1. La gráfica de los datos (en la figura 6.2) sugiere que no hay tendencia o patrón estacional. Es posible también que el nivel (o media) pudiera estar cambiando lentamente a lo largo del tiempo.  El problema pide ajustar un modelo de suavización exponencial simple (SES) a la serie de capturas de bacalao, ya que la gráfica sugiere ausencia de tendencia y estacionalidad, pero con posibilidad de que el nivel cambie lentamente. Ese es exactamente el escenario de SES.

**¿Qué nos dice esto?**

1.  **No hay tendencia:** La serie no va claramente hacia arriba ni hacia abajo de forma sostenida.
2.  **No hay estacionalidad:** No hay picos que se repitan cada 12 meses (a pesar de ser datos mensuales).
3.  **Nivel cambiante:** La media del proceso no es perfectamente constante.

**Conclusión:** Este es el escenario perfecto para aplicar un modelo de **Suavización Exponencial Simple (SES)**. Este método es ideal para series sin tendencia ni estacionalidad, ya que su objetivo es "suavizar" las fluctuaciones aleatorias para estimar el nivel actual, dándole más peso a las observaciones recientes que a las pasadas.

**TABLA 6.1 Pesca de bacalao (en toneladas)**

| Mes        | Año 1 | Año 2 |
| ---------- | ----- | ----- |
| Enero      | 362   | 276   |
| Febrero    | 381   | 334   |
| Marzo      | 317   | 394   |
| Abril      | 297   | 334   |
| Mayo       | 399   | 384   |
| Junio      | 402   | 314   |
| Julio      | 375   | 344   |
| Agosto     | 349   | 337   |
| Septiembre | 386   | 345   |
| Octubre    | 328   | 362   |
| Noviembre  | 389   | 314   |
| Diciembre  | 343   | 365   |

### 📐 El Modelo Matemático (SES)

Para cada periodo **t**:

- **Nivel suavizado:** $L_t = \alpha Y_t + (1 - \alpha)(L_{t-1})$
- **Pronóstico un paso adelante:** $\hat{Y}_t = L_{t-1}$
- **Error de pronóstico:** $e_t = Y_t - \hat{Y}_t$

Dónde:
 -  $\hat{Y}_t$ ​es el dato real de pesca en el mes *t*.
 -  $\alpha$ es la constante de suavización $(0<\alpha<1)$
 -  $L_{0}$ ​es el valor inicial (usualmente el promedio de los primeros datos, en este caso, el promedio del Año 1).

### 📊 ¿Qué esperar de los resultados?

1.  **El** **α** **óptimo:** Al correr la búsqueda, verás que el *α* que minimiza el error suele estar entre **0.1 y 0.3**. Esto indica que el nivel cambia, pero de forma relativamente suave (el modelo le da peso tanto a los datos recientes como a la historia pasada).

2.  **El Pronóstico:** Para la Suavización Exponencial Simple, el pronóstico de todos los meses futuros (mes 25, 26, etc.) será exactamente igual al último nivel calculado (*L*<sub>24</sub>​). La línea azul del gráfico terminará siendo una línea horizontal hacia el futuro.

3.  **La tabla (`D_opt`):** Tendrá las columnas `Periodo`, `Bacalao`, `Lt` (el nivel estimado), `Pt` (el pronóstico que se hizo un mes antes), `Et` (diferencia real - pronóstico) y `Et2` (error al cuadrado).

## ✅ Modelo correcto de SES

Para `t = 2, …, n`:

$$
L_{t} = \alpha y_t + (1 - \alpha) L_{t-1}, \quad \hat{y}_t = L_{t-1}, \quad e_t = y_t - \hat{y}_t
$$

con $L_1$ = $L_0$ (inicialización). El pronóstico a `h` pasos es **constante**: $\hat{y}_{n+h} = L_{n}$.

In [7]:
Periodo<-seq(0,24)
Bacalao <- c(362, 381, 317, 297, 399, 402, 375, 349, 386, 328, 389, 343, # Año 1
             276, 334, 394, 334, 384, 314, 344, 337, 345, 362, 314, 365) # Año 2
n <- length(Bacalao) # n será igual a 24
L <- 12
Lt[1] <- L0
L0<-(sum(Bacalao[1:(L+1)])/L)
Lt  <- numeric(n)
Pt  <- numeric(n)
Et  <- numeric(n)
Et2 <- numeric(n)

Para alfa=0.1

In [8]:
# Función SES
ses <- function(y, alpha, L0) {
  n <- length(y)
  Lt <- Pt <- Et <- numeric(n)
  Lt[1] <- L0
  for (i in 2:n) {
    Lt[i] <- alpha*y[i] + (1-alpha)*Lt[i-1]
    Pt[i] <- Lt[i-1]
    Et[i] <- y[i] - Pt[i]
  }
  return(list(Lt=Lt, Pt=Pt, Et=Et, SSE=sum(Et^2)))
}

## 🔎 Para `alfa1 = 0.034` (suavizado **muy bajo**):

-   `Lt` cambia muy lentamente → la serie suavizada queda casi plana alrededor de `L0 ≈ 352.58`.
-   `Pt[i] ≈ L0` para casi todas las `i`, porque el nivel apenas se actualiza.
-   El error `Et[i] ≈ Bacalao[i] − L0` refleja casi directamente la **desviación bruta** de cada observación respecto del nivel inicial, en lugar de un error de pronóstico uno-paso-adelante bien adaptado.
-   Comparado con `alfa = 0.1`, el ajuste es **mucho más conservador** y reacciona muy poco a la dinámica real de la serie.

### Sugerencia metodológica

El valor `alfa1 = 0.034` corresponde aproximadamente a `2/(L+1)` con `L ≈ 58`, muy distinto al `L = 12` que usás para inicializar. Si el objetivo es consistencia, se suele elegir `α` en función del mismo `L` usado para `L0` (por ejemplo, `α = 2/(L+1) ≈ 0.154` para `L = 12`).

## 🔎 Interpretación crítica del resultado

Con `alfa1 = 0.034`, los valores que obtenés son **muy altos**:

-   `SSE ≈ 26 000`
-   `S ≈ 34`

Esto ocurre porque `alfa1 = 0.034` es **demasiado pequeño**: el nivel `Lt` apenas se mueve de `L0 ≈ 352.58` y el pronóstico `Pt` es prácticamente **constante** en `≈ 352.58`. En la práctica el modelo **predice la media** en cada paso.

> Comparación: el desvío estándar de `Bacalao` (sin el primer `0`) es ≈ `33–34`. Es decir, `S ≈ sd(Bacalao)`. **El modelo no está aportando información**: el error de pronóstico es del mismo tamaño que la variabilidad total de la serie.

In [ ]:
alfa1<-0.034
Lt[1]<-L0
for (i in 2:n)
{
  Lt[i]<-alfa1*Bacalao[i]+(1-alfa1)*Lt[i-1]
  Pt[i]<-Lt[i-1]
  Et[i]<-(Bacalao[i:(n-1)]-Pt[i:(n-1)])
  Et2[i]<-((Et[i])^2)
}
E<-data.frame(Periodo,Bacalao,Lt,Pt,Et,Et2)
E

Warning message in Et[i] <- (Bacalao[i:(n - 1)] - Pt[i:(n - 1)]):
“number of items to replace is not a multiple of replacement length”
Warning message in Et[i] <- (Bacalao[i:(n - 1)] - Pt[i:(n - 1)]):
“number of items to replace is not a multiple of replacement length”
Warning message in Et[i] <- (Bacalao[i:(n - 1)] - Pt[i:(n - 1)]):
“number of items to replace is not a multiple of replacement length”
Warning message in Et[i] <- (Bacalao[i:(n - 1)] - Pt[i:(n - 1)]):
“number of items to replace is not a multiple of replacement length”
Warning message in Et[i] <- (Bacalao[i:(n - 1)] - Pt[i:(n - 1)]):
“number of items to replace is not a multiple of replacement length”
Warning message in Et[i] <- (Bacalao[i:(n - 1)] - Pt[i:(n - 1)]):
“number of items to replace is not a multiple of replacement length”
Warning message in Et[i] <- (Bacalao[i:(n - 1)] - Pt[i:(n - 1)]):
“number of items to replace is not a multiple of replacement length”
Warning message in Et[i] <- (Bacalao[i:(n - 1)] - Pt[i:

Periodo,Bacalao,Lt,Pt,Et,Et2
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
0,0,360.6667,0.0000,0.000000,0.000000
1,362,360.7120,360.6667,1.333333,1.777778
2,381,361.4018,360.7120,20.288000,411.602944
3,317,359.8921,361.4018,-44.401792,1971.519133
4,297,357.7538,359.8921,-62.892131,3955.420151
5,399,359.1562,357.7538,41.246201,1701.249129
6,402,360.6129,359.1562,42.843831,1835.593815
7,375,361.1020,360.6129,14.387140,206.989806
8,349,360.6906,361.1020,-12.102022,146.458948


Error estándar SSE

In [ ]:
t<-24
SSE<-sum(Et2)
S<-sqrt(SSE/(t-1))

In [ ]:
S

[1] 34.94665